In [1]:
import pandas as pd
import numpy as np
import os
import glob
import importlib
import config
import yfinance as yf

importlib.reload(config)

pd.options.display.float_format = '{:,.4f}'.format
pd.set_option('display.max_columns', None)

In [2]:
def clean_id_number(column):
    """
    Convert Indonesian formatted strings (e.g., '1.500,50') to numeric floats.
    
    Args:
        column (pd.Series): Raw string/object column from IDX Excel.
    Returns:
        pd.Series: Cleaned float64 numeric series.
    """
    # Standardize separators: Remove dots (thousands) and fix commas (decimals)
    clean_col = (column.astype(str)
                 .str.strip()
                 .str.replace('.', '', regex=False)
                 .str.replace(',', '.', regex=False)
                 .str.replace(r'[^\d.]', '', regex=True))
    
    return pd.to_numeric(clean_col, errors='coerce')

In [3]:
def convert_idx_date(df, date_col):
    """
    Converts Indonesian date strings (e.g., '01 Des 2025') to datetime objects.
    """
    id_to_en_months = {
        'Jan': 'Jan', 'Feb': 'Feb', 'Mar': 'Mar', 'Apr': 'Apr',
        'Mei': 'May', 'Jun': 'Jun', 'Jul': 'Jul', 'Agu': 'Aug',
        'Sep': 'Sep', 'Okt': 'Oct', 'Nov': 'Nov', 'Des': 'Dec'
    }
    
    # Replace months using the dictionary
    for id_mon, en_mon in id_to_en_months.items():
        df[date_col] = df[date_col].str.replace(id_mon, en_mon, regex=False)
    
    # Convert to actual datetime objects
    df[date_col] = pd.to_datetime(df[date_col], format='%d %b %Y', errors='coerce')
    return df

In [4]:
def calculate_investment_metrics(df):
    """
    Calculate return (Y), foreign intensity (X1), and volume (X2) metrics.
    
    Converts key columns to float to ensure precision and prevent rounding errors.
    """
    cols = ['Penutupan', 'Sebelumnya', 'Foreign Buy', 'Foreign Sell', 'Volume']
    df = df.astype({col: float for col in cols if col in df.columns})

    # Calculations
    df['Y_Price_Return'] = (df['Penutupan'] - df['Sebelumnya']) / df['Sebelumnya'].replace(0, np.nan)
    df['X1_Foreign_Intensity'] = (df['Foreign Buy'] - df['Foreign Sell']) / (df['Volume'] + 1)
    df['X2_Volume'] = df['Volume']

    # --- Handling NaNs here ---
    # We fill metrics with 0 because if data is missing, we assume no change/flow.
    metric_cols = ['Y_Price_Return', 'X1_Foreign_Intensity', 'X2_Volume']
    df[metric_cols] = df[metric_cols].fillna(0)
    
    return df

In [5]:
"""
Portfolio Universe Filtering
Focus: Mid-Cap growth and value hunters.
Strategy: Filters Indonesian stocks listed on the 'Utama' and 'Pengembangan' boards.
"""

# 1. Load Summary Trading Stocks and List Stocks (in masked folder)

input_daftar = os.path.join(config.MASKED_DATA_DAFTAR_SAHAM_PATH, "Daftar_Saham_Masked.csv")
input_summary_trade = os.path.join(config.MASKED_DATA_RINGKASAN_PERDAGANGAN_SAHAM_PATH, "Master_Masked_Data.csv")

try:
    df_daftar = pd.read_csv(input_daftar)
    df_trade = pd.read_csv(input_summary_trade)
    
    df_daftar = df_daftar.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
    df_trade = df_trade.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
    
    print(f"Successfully read: {os.path.basename(input_daftar)}")
    print(df_daftar.head())
    print(f"\n{'-' * 80}")
    print(f"Successfully read: {os.path.basename(input_summary_trade)}")
    print(df_trade.head())

except Exception as e:
    print(f"An error occurred: {e}")

Successfully read: Daftar_Saham_Masked.csv
   No       Kode           Saham   Papan Pencatatan
0   1  Saham_001   1.924.688.333              Utama
1   2  Saham_002   3.935.892.857  Pemantauan Khusus
2   3  Saham_003     620.806.680  Pemantauan Khusus
3   4  Saham_004   2.753.165.000              Utama
4   5  Saham_005  17.120.389.700              Utama

--------------------------------------------------------------------------------
Successfully read: Master_Masked_Data.csv
   No Kode Saham  Sebelumnya  Open Price Tanggal Perdagangan Terakhir  \
0   1  Saham_891        7500        7525                  01 Des 2025   
1   2  Saham_001        7575        7575                  01 Des 2025   
2   3  Saham_002          55           0                  01 Des 2025   
3   4  Saham_003        2690           0                  01 Des 2025   
4   5  Saham_004        2920        2920                  01 Des 2025   

   First Trade  Tertinggi  Terendah  Penutupan  Selisih   Volume        Nilai  \
0

In [6]:
# 2. Start the selection

unique_boards = df_daftar['Papan Pencatatan'].unique()
print("--- Unique Listing Boards ---")
print(unique_boards)

--- Unique Listing Boards ---
['Utama' 'Pemantauan Khusus' 'Pengembangan' 'Akselerasi' 'Ekonomi Baru']


In [7]:
# 3. Board Selection (Universe Gatekeeper)
# We prioritize 'Utama' and 'Pengembangan' boards to ensure we analyze established 
# companies, effectively filtering out high-risk speculative or transitionary tiers.

unique_boards_selection = ['Utama', 'Pengembangan']
stocks_selection = df_daftar[df_daftar['Papan Pencatatan'].isin(unique_boards_selection)].copy()

print(f"\n{'='*30} GATEKEEPER {'='*30}\n")
print(stocks_selection.head())

print(f"\n{'-' * 30} DATA SCHEMA {'-' * 30}")
print(stocks_selection.info())


============================== GATEKEEPER ==============================

   No       Kode           Saham Papan Pencatatan
0   1  Saham_001   1.924.688.333            Utama
3   4  Saham_004   2.753.165.000            Utama
4   5  Saham_005  17.120.389.700            Utama
5   6  Saham_006  17.675.160.000     Pengembangan
6   7  Saham_007     589.896.800     Pengembangan

------------------------------ DATA SCHEMA ------------------------------
<class 'pandas.core.frame.DataFrame'>
Index: 741 entries, 0 to 952
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   No                741 non-null    int64 
 1   Kode              741 non-null    object
 2   Saham             741 non-null    object
 3   Papan Pencatatan  741 non-null    object
dtypes: int64(1), object(3)
memory usage: 28.9+ KB
None


In [8]:
# 5. Numerical Data Sanitization
# Applying the custom 'clean_id_number' utility to standardize the 'Saham' column.
# This ensures total shares outstanding are represented as numeric types for calculation.

stocks_selection['Saham'] = clean_id_number(stocks_selection['Saham'])

print(f"Verified Dtype for Saham: {stocks_selection['Saham'].dtype}")

Verified Dtype for Saham: int64


In [9]:
# 6. Check df_trade

print(df_trade.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55547 entries, 0 to 55546
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   No                            55547 non-null  int64  
 1   Kode Saham                    55547 non-null  object 
 2   Sebelumnya                    55547 non-null  int64  
 3   Open Price                    55547 non-null  int64  
 4   Tanggal Perdagangan Terakhir  55547 non-null  object 
 5   First Trade                   55547 non-null  int64  
 6   Tertinggi                     55547 non-null  int64  
 7   Terendah                      55547 non-null  int64  
 8   Penutupan                     55547 non-null  int64  
 9   Selisih                       55547 non-null  int64  
 10  Volume                        55547 non-null  int64  
 11  Nilai                         55547 non-null  int64  
 12  Frekuensi                     55547 non-null  int64  
 13  I

In [10]:
# 7. Relational Database Integration
# Merging daily trade data with the 'Gatekeeper' data to enrich the dataset 
# with Share Counts and Listing Board information. We use an 'inner' join 

df_trade = convert_idx_date(df_trade, 'Tanggal Perdagangan Terakhir')

stocks_merge = (pd.merge(
    df_trade, stocks_selection[['Kode', 'Saham', 'Papan Pencatatan']], 
    left_on='Kode Saham', right_on='Kode', 
    how='inner'
)
.drop(columns=['Kode'])
.sort_values(by=['Tanggal Perdagangan Terakhir', 'Kode Saham'])
.reset_index(drop=True)
.copy())

print(f"{'='*20} MERGE VERIFICATION {'='*20}\n")
print(stocks_merge[['Tanggal Perdagangan Terakhir', 'Kode Saham', 'Penutupan', 'Saham', 'Papan Pencatatan']]
      .assign(Penutupan=lambda x: x['Penutupan'].astype(float),
              Saham=lambda x: x['Saham'].astype(float))
      .head())

==================== MERGE VERIFICATION ====================

  Tanggal Perdagangan Terakhir Kode Saham   Penutupan               Saham  \
0                   2025-12-01  Saham_001  7,650.0000  1,924,688,333.0000   
1                   2025-12-01  Saham_004  2,910.0000  2,753,165,000.0000   
2                   2025-12-01  Saham_005    416.0000 17,120,389,700.0000   
3                   2025-12-01  Saham_006    133.0000 17,675,160,000.0000   
4                   2025-12-01  Saham_007 15,375.0000    589,896,800.0000   

  Papan Pencatatan  
0            Utama  
1            Utama  
2            Utama  
3     Pengembangan  
4     Pengembangan  


In [11]:
# 8. Valuation Engine: Market Capitalization Calculation
# We derive the Market Cap by multiplying the closing price by the total shares 
# outstanding. This metric is essential for our Mid-Cap segmentation strategy.

stocks_merge['Market Cap'] = stocks_merge['Penutupan'] * stocks_merge['Saham']

stocks_merge = stocks_merge.dropna(subset=['Market Cap']).copy()

print(f"{'='*20} MARKET CAP VERIFICATION {'='*20}\n")
print(stocks_merge[['Kode Saham', 'Penutupan', 'Saham', 'Market Cap']]
      .astype({'Penutupan': float, 'Saham': float, 'Market Cap': float})
      .head())

==================== MARKET CAP VERIFICATION ====================

  Kode Saham   Penutupan               Saham              Market Cap
0  Saham_001  7,650.0000  1,924,688,333.0000 14,723,865,747,450.0000
1  Saham_004  2,910.0000  2,753,165,000.0000  8,011,710,150,000.0000
2  Saham_005    416.0000 17,120,389,700.0000  7,122,082,115,200.0000
3  Saham_006    133.0000 17,675,160,000.0000  2,350,796,280,000.0000
4  Saham_007 15,375.0000    589,896,800.0000  9,069,663,300,000.0000


In [12]:
# 9. Strategic Universe Segmentation (Mid-Cap Filtering)
# We isolate the Mid-Cap segment (2T - 20T IDR) to align with our 
# 'Value and Growth' mandate. This filter excludes high-volatility 
# Small-Caps and over-saturated Big-Caps.

cap_min = 2_000_000_000_000 
cap_max = 20_000_000_000_000

stocks_mid_cap = stocks_merge[
    (stocks_merge['Market Cap'] >= cap_min) & 
    (stocks_merge['Market Cap'] <= cap_max)
].copy()

print(f"Universe filtered from {len(stocks_merge)} to {len(stocks_mid_cap)} tickers.")
print(f"Mid-Cap filter successful.")

print(stocks_mid_cap[['Kode Saham', 'Market Cap']]
      .sort_values('Market Cap', ascending=False)
      .astype({'Market Cap': float}))

Universe filtered from 42961 to 14882 tickers.
Mid-Cap filter successful.
      Kode Saham              Market Cap
22417  Saham_124 19,999,824,312,500.0000
22418  Saham_124 19,999,824,312,500.0000
33901  Saham_752 19,986,978,375,000.0000
36716  Saham_539 19,975,735,000,000.0000
27342  Saham_894 19,955,645,430,000.0000
...          ...                     ...
32077  Saham_284  2,000,453,281,760.0000
13703  Saham_479  2,000,000,100,000.0000
40597  Saham_790  2,000,000,000,000.0000
35410  Saham_790  2,000,000,000,000.0000
37633  Saham_790  2,000,000,000,000.0000

[14882 rows x 2 columns]


In [14]:
# 10. Quantitative Logic Validation (The "Truth Test")
# Before executing the bulk processing loop, we validate the metric engine 
# on a sample subset to ensure floating-point precision and catch any 
# potential 'zero-rounding' or 'division by zero' errors.

master_data = calculate_investment_metrics(stocks_mid_cap.copy())

master_data = master_data[master_data['Penutupan'] > 0]

print(f"{'='*20} REVIEW CALCULATION {'='*20}\n")
print(master_data[['Kode Saham', 'Penutupan', 'Sebelumnya', 'Y_Price_Return', 'X1_Foreign_Intensity']]
    .head(10))

print(f"\nMean of Price Return: {master_data['Y_Price_Return'].mean():.6f}")
print(f"Mean of Foreign Intensity: {master_data['X1_Foreign_Intensity'].mean():.6f}")

==================== REVIEW CALCULATION ====================

   Kode Saham   Penutupan  Sebelumnya  Y_Price_Return  X1_Foreign_Intensity
0   Saham_001  7,650.0000  7,575.0000          0.0099                0.3167
1   Saham_004  2,910.0000  2,920.0000         -0.0034               -0.1765
2   Saham_005    416.0000    418.0000         -0.0048                0.0174
3   Saham_006    133.0000    139.0000         -0.0432               -0.0017
4   Saham_007 15,375.0000 15,350.0000          0.0016                0.0000
5   Saham_008    244.0000    250.0000         -0.0240                0.1497
16  Saham_022  1,810.0000  1,850.0000         -0.0216               -0.0359
19  Saham_025  1,310.0000  1,525.0000         -0.1410               -0.0706
22  Saham_028     99.0000     98.0000          0.0102                0.0617
23  Saham_029  1,045.0000  1,055.0000         -0.0095                0.0000

Mean of Price Return: 0.001532
Mean of Foreign Intensity: -0.011059


In [15]:
# 11. Calculate IHSG Return for X3
try:
    ihsg_data = yf.download('^JKSE', start='2025-01-01', end='2026-03-26')
    ihsg_data.columns = ihsg_data.columns.get_level_values(0)
    df_ihsg = ihsg_data.reset_index().copy()
    df_ihsg['X3_IHSG_Return'] = df_ihsg['Close'].pct_change()
    df_ihsg['Date'] = pd.to_datetime(df_ihsg['Date']).dt.normalize()
    
    print("✅ IHSG data successfully downloaded from Yahoo Finance!")
    print(df_ihsg[['Date', 'Close', 'X3_IHSG_Return']].tail())

except Exception as e:
    print(f"❌ Failed to download IHSG data: {e}")
    
master_clean_data = (pd.merge(
    master_data, df_ihsg[['Date', 'X3_IHSG_Return']], 
    left_on='Tanggal Perdagangan Terakhir', right_on='Date', 
    how='left'
)
.drop(columns=['Date'])
.copy())

[*********************100%***********************]  1 of 1 completed

✅ IHSG data successfully downloaded from Yahoo Finance!
Price       Date      Close  X3_IHSG_Return
282   2026-03-12 7,362.1172         -0.0037
283   2026-03-13 7,137.2119         -0.0305
284   2026-03-16 7,022.2881         -0.0161
285   2026-03-17 7,106.8389          0.0120
286   2026-03-25 7,302.1211          0.0275


In [16]:
# 12. Final Quantitative Audit
# Validating the consolidated universe size and performing a statistical 
# sanity check on core investment metrics (Y, X1, X2)
# displaying the synchronized final states

print(f"{'='*20} MASTER DATA ARCHITECTURE {'='*20}")
print(f"Final Size: {master_clean_data['Kode Saham'].nunique()} Tickers")
print(f"Total Data Points: {len(master_clean_data)} rows")

print(f"\n{'='*20} MASTER HEAD VIEW {'='*20}")
print(master_clean_data.head())

==================== MASTER DATA ARCHITECTURE ====================
Final Size: 301 Tickers
Total Data Points: 14882 rows

==================== MASTER HEAD VIEW ====================
   No Kode Saham  Sebelumnya  Open Price Tanggal Perdagangan Terakhir  \
0   2  Saham_001  7,575.0000        7575                   2025-12-01   
1   5  Saham_004  2,920.0000        2920                   2025-12-01   
2   6  Saham_005    418.0000         418                   2025-12-01   
3   8  Saham_006    139.0000         140                   2025-12-01   
4  10  Saham_007 15,350.0000       15500                   2025-12-01   

   First Trade  Tertinggi  Terendah   Penutupan  Selisih          Volume  \
0         7575       7650      7525  7,650.0000       75    755,600.0000   
1         2910       2930      2900  2,910.0000      -10    633,500.0000   
2          420        422       416    416.0000       -2 69,207,600.0000   
3          140        141       130    133.0000       -6  9,585,600.0000   


In [17]:
# 13. Save the Clean Data

output_path_clean = os.path.join(config.CLEAN_MASTER_DATA_PATH, "clean_master_data.csv")
master_clean_data.to_csv(output_path_clean, index=False)

print(f"clean master file saved to: {output_path_clean}")

clean master file saved to: D:\Data Science\saham exercise\Project 1\1_Data\3_Cleaning_Data\clean_master_data.csv
